# 📊 Análisis Iterativo de Modelos de Series de Tiempo
## Riesgo País Argentino — VECM · VAR · ARDL

**Flujo de trabajo:**
1. Leer `Variables Regresivas/analisis_series_*.csv` (generado desde el visualizador)
2. Parsear el `CATALOG` de `visualizador_template.html` para obtener rutas y columnas de cada serie
3. Cargar datos desde `data/Variables Finales/` con la misma lógica que el visualizador (resample = media, deflactores idénticos)
4. Ejecutar pre-tests (ADF/KPSS, Granger, Johansen)
5. Estimar ARDL · VAR · VECM sobre **todas las combinaciones posibles** de variables explicativas
6. Visualizar coeficientes y criterios de información

---
**Solo editar la Sección 3 (CONFIGURACIÓN).**

## 1. Instalación e Imports

In [ ]:
import subprocess, sys, os, re

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for _pkg in ['statsmodels', 'scipy', 'scikit-learn', 'openpyxl', 'matplotlib', 'numpy', 'pandas', 'ipywidgets']:
    try:
        __import__(_pkg.replace('-', '_'))
    except ImportError:
        print(f'Instalando {_pkg}...')
        _install(_pkg)

%matplotlib inline

import warnings
import glob
import random
from itertools import combinations
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))
from src.utils import test_estacionariedad, test_causalidad_granger, resumen_estadistico

print('Módulos cargados.')


## 2. Funciones Auxiliares

In [2]:
# ── Directorios base (igual que generar_visualizador.py) ─────────────────────
VARS_DIR = (Path('..') / 'data' / 'Variables Finales').resolve()
RAW_DIR  = (Path('..') / 'data' / 'raw').resolve()

# ── 1. Parsear CATALOG desde visualizador_template.html ──────────────────────
def _parse_catalog():
    for _cand in [Path('visualizador_template.html'),
                  Path('notebooks') / 'visualizador_template.html']:
        if _cand.exists():
            html_path = _cand
            break
    else:
        raise FileNotFoundError('No se encontró visualizador_template.html')

    with open(html_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Acotar búsqueda solo a la sección CATALOG (evita falsos matches en JS posterior)
    _start = content.find('const CATALOG = [')
    _end   = content.find('\n];', _start)
    catalog_txt = content[_start:_end + 3] if _start >= 0 and _end >= 0 else content

    cat = {}
    _PREF    = {'D': 0, 'W': 1, 'M': 2, 'Q': 3, 'A': 4}
    _FREQ_RE = re.compile(r"freq\s*:\s*'([^']+)'")

    # ── Paso 1: ítems de fuente única (todos los campos en una sola línea) ────
    _SINGLE = re.compile(
        r"id\s*:\s*'([^']+)'.*?label\s*:\s*'([^']+)'.*?"
        r"file\s*:\s*'([^']+)'.*?dateCol\s*:\s*'([^']+)'.*?valCol\s*:\s*'([^']+)'"
    )
    for line in catalog_txt.splitlines():
        m = _SINGLE.search(line)
        if not m:
            continue
        cid, label, file_, dc, vc = m.groups()
        if label in cat:
            continue
        fm = _FREQ_RE.search(line)
        freq = fm.group(1) if fm else 'D'
        cat[label] = {'id': cid, 'file': file_, 'dateCol': dc, 'valCol': vc, 'freq': freq}

    # ── Paso 2: ítems multi-fuente (sources:[...] en varias líneas) ───────────
    _MULTI = re.compile(
        r"\{\s*id\s*:\s*'([^']+)'\s*,\s*label\s*:\s*'([^']+)'[^[]*?"
        r"sources\s*:\s*\[(.*?)\]",
        re.DOTALL
    )
    _SRC = re.compile(
        r"\{\s*freq\s*:\s*'([^']+)'[^}]*?"
        r"file\s*:\s*'([^']+)'[^}]*?"
        r"dateCol\s*:\s*'([^']+)'[^}]*?"
        r"valCol\s*:\s*'([^']+)'"
    )
    for m in _MULTI.finditer(catalog_txt):
        cid, label, srcs_txt = m.groups()
        if label in cat:
            continue
        srcs = [{'freq': sm.group(1), 'file': sm.group(2),
                 'dateCol': sm.group(3), 'valCol': sm.group(4)}
                for sm in _SRC.finditer(srcs_txt)]
        if not srcs:
            continue
        best = sorted(srcs, key=lambda s: _PREF.get(s['freq'], 99))[0]
        cat[label] = {
            'id'     : cid,
            'file'   : best['file'],
            'dateCol': best['dateCol'],
            'valCol' : best['valCol'],
            'freq'   : best['freq'],
            'sources': srcs,
        }

    return cat


CATALOG_VIZ = _parse_catalog()
print(f'✅ CATALOG parseado: {len(CATALOG_VIZ)} series')


# ── 2. Detección automática de frecuencia ─────────────────────────────────────
def detectar_frecuencia(df_csv):
    _ORD = {'D': 0, 'W': 1, 'M': 2, 'Q': 3, 'A': 4}
    finest = []
    for _, row in df_csv.iterrows():
        for col in ['Serie A', 'Serie B']:
            nombre = str(row.get(col, '') or '').strip()
            if not nombre or nombre in ('', '—', '-', 'nan'):
                continue
            if nombre not in CATALOG_VIZ:
                continue
            meta = CATALOG_VIZ[nombre]
            if 'sources' in meta:
                src_freqs = [s['freq'] for s in meta['sources']]
                finest.append(min(src_freqs, key=lambda f: _ORD.get(f, 99)))
            else:
                finest.append(meta.get('freq', 'D'))
    if not finest:
        return 'M'
    return max(finest, key=lambda f: _ORD.get(f, -1))


# ── 3. Resampling (media por período, igual que el visualizador JS) ───────────
def _resample_mean(serie: pd.Series, to_freq: str) -> pd.Series:
    _MAP = {'D': 'D', 'W': 'W', 'M': 'ME', 'Q': 'QE', 'A': 'YE'}
    pf = _MAP.get(to_freq, to_freq)
    try:
        return serie.resample(pf).mean()
    except ValueError:
        _MAP_OLD = {'D': 'D', 'W': 'W', 'M': 'M', 'Q': 'Q', 'A': 'A'}
        return serie.resample(_MAP_OLD.get(to_freq, to_freq)).mean()


# ── 4. Carga genérica ─────────────────────────────────────────────────────────
def _buscar_archivo(*rutas_relativas):
    for ruta in rutas_relativas:
        for base in [VARS_DIR, RAW_DIR]:
            p = base / ruta
            if p.exists():
                return p
    return None


def _match_col(wanted: str, available_cols) -> str | None:
    ws = wanted.strip()
    if ws in available_cols:
        return ws
    if wanted in available_cols:
        return wanted
    return None


def _leer_serie_csv(path: Path, col_fecha: str, col_valor: str) -> pd.Series:
    df = pd.read_csv(path, low_memory=False, encoding='utf-8-sig')
    df.columns = df.columns.str.strip()
    dc = _match_col(col_fecha, df.columns)
    vc = _match_col(col_valor, df.columns)
    if dc is None:
        raise KeyError(f'col_fecha "{col_fecha}" no encontrada en {path.name}. '
                       f'Cols: {list(df.columns[:8])}')
    if vc is None:
        raise KeyError(f'col_valor "{col_valor}" no encontrada en {path.name}. '
                       f'Cols: {list(df.columns[:8])}')
    df['_f'] = pd.to_datetime(df[dc], errors='coerce')
    df = df.dropna(subset=['_f']).set_index('_f').sort_index()
    return pd.to_numeric(df[vc], errors='coerce').dropna()


# ── 5. Deflactores ────────────────────────────────────────────────────────────
_cache_pbi = _cache_cer = _cache_tip = _cache_mep = None

def _get_pbi():
    global _cache_pbi
    if _cache_pbi is not None: return _cache_pbi
    p = _buscar_archivo('pbi_constante_2004.csv')
    if p is None: print('  ⚠️  pbi_constante_2004.csv no encontrado'); return None
    _cache_pbi = _leer_serie_csv(p, 'fecha', 'pbi'); return _cache_pbi

def _get_cer():
    global _cache_cer
    if _cache_cer is not None: return _cache_cer
    p = _buscar_archivo('cer.csv', 'bcra/cer.csv')
    if p is None: print('  ⚠️  cer.csv no encontrado'); return None
    _cache_cer = _leer_serie_csv(p, 'fecha', 'cer'); return _cache_cer

def _get_tip():
    global _cache_tip
    if _cache_tip is not None: return _cache_tip
    p = _buscar_archivo('tip_etf.csv', 'global/tip_etf.csv')
    if p is None: print('  ⚠️  tip_etf.csv no encontrado'); return None
    _cache_tip = _leer_serie_csv(p, 'fecha', 'tip_close'); return _cache_tip

def _get_mep():
    global _cache_mep
    if _cache_mep is not None: return _cache_mep
    p = _buscar_archivo('brecha_cambiaria.csv')
    if p is None: print('  ⚠️  brecha_cambiaria.csv no encontrado'); return None
    _cache_mep = _leer_serie_csv(p, 'fecha', 'mep'); return _cache_mep


def _alinear_deflactor(raw: pd.Series, frecuencia: str,
                        target_index: pd.DatetimeIndex) -> pd.Series | None:
    """
    Resamplea `raw` a `frecuencia`, luego alinea con `target_index`.
    Maneja datos de baja frecuencia (ej. PBI trimestral) rellenando con ffill+bfill
    para que no queden NaN en el rango de análisis.
    """
    al = _resample_mean(raw, frecuencia)
    # Para frecuencias más bajas que la de la serie (ej. PBI trim → mensual),
    # solo quedan datos en los períodos exactos del deflactor.
    # reindex+ffill propaga cada valor hasta el siguiente período.
    al = al.reindex(target_index, method='ffill').ffill().bfill()
    if al.isna().all():
        return None
    return al


def _aplicar_deflactor(serie: pd.Series, deflactor_name: str,
                        frecuencia: str) -> pd.Series:
    if not deflactor_name or str(deflactor_name).strip() in ('', '—', '-', 'nan'):
        return serie
    for d in str(deflactor_name).split(','):
        d = d.strip().upper()
        if not d or d in ('—', '-'): continue

        if d == 'PBI':
            raw = _get_pbi()
            if raw is None: continue
            al = _alinear_deflactor(raw, frecuencia, serie.index)
            if al is None:
                print('  ⚠️  PBI: sin superposición de fechas con la serie'); continue
            fp = al.iloc[0]
            serie = serie * fp / al

        elif d == 'CER':
            raw = _get_cer()
            if raw is None: continue
            al = _alinear_deflactor(raw, frecuencia, serie.index)
            if al is None:
                print('  ⚠️  CER: sin superposición de fechas con la serie'); continue
            lc = al.iloc[-1]
            serie = serie / al * lc

        elif d in ('TIP', 'IPC'):
            raw = _get_tip()
            if raw is None: continue
            al = _alinear_deflactor(raw, frecuencia, serie.index)
            if al is None:
                print(f'  ⚠️  {d}: sin superposición de fechas con la serie'); continue
            lt = al.iloc[-1]
            serie = serie / al * lt

        elif d == 'MEP':
            raw = _get_mep()
            if raw is None: continue
            al = _alinear_deflactor(raw, frecuencia, serie.index)
            if al is None:
                print('  ⚠️  MEP: sin superposición de fechas con la serie'); continue
            serie = serie / al

        else:
            print(f'    ⚠️  Deflactor no reconocido: "{d}"')
    return serie


# ── 6. Carga de serie desde CATALOG ──────────────────────────────────────────
def _cargar_serie_catalog(label: str, frecuencia: str,
                           fecha_inicio: str = None, fecha_fin: str = None):
    """
    Carga la serie `label` (nombre exacto del CATALOG) en la frecuencia indicada.
    Devuelve pd.Series o None si no se encuentra / hay error.
    """
    if label not in CATALOG_VIZ:
        print(f'  ⚠️  "{label}" no encontrado en CATALOG ({len(CATALOG_VIZ)} series).')
        return None

    meta  = CATALOG_VIZ[label]
    _PREF = {'D': 0, 'W': 1, 'M': 2, 'Q': 3, 'A': 4}

    if 'sources' in meta:
        srcs  = meta['sources']
        exact = [s for s in srcs if s['freq'] == frecuencia]
        src   = exact[0] if exact else sorted(srcs, key=lambda s: _PREF.get(s['freq'], 99))[0]
        file_, dc, vc = src['file'], src['dateCol'], src['valCol']
    else:
        file_, dc, vc = meta['file'], meta['dateCol'], meta['valCol']

    file_path = VARS_DIR / file_
    if not file_path.exists():
        file_path_raw = RAW_DIR / file_
        if file_path_raw.exists():
            file_path = file_path_raw
        else:
            print(f'  ⚠️  Archivo no encontrado: {VARS_DIR / file_}')
            return None

    try:
        if str(file_).endswith(('.xlsx', '.xls')):
            sheet = 'Unificado' if 'fiscal' in str(file_).lower() else 0
            df = pd.read_excel(file_path, sheet_name=sheet)
        else:
            df = pd.read_csv(file_path, low_memory=False, encoding='utf-8-sig')
    except Exception as e:
        print(f'  ⚠️  Error leyendo {file_}: {e}')
        return None

    df.columns = df.columns.str.strip()
    dc_match = _match_col(dc, df.columns)
    vc_match = _match_col(vc, df.columns)

    if dc_match is None:
        print(f'  ⚠️  dateCol "{dc}" no hallada en {file_}. Cols: {list(df.columns[:8])}')
        return None
    if vc_match is None:
        print(f'  ⚠️  valCol "{vc}" no hallada en {file_}. Cols: {list(df.columns[:8])}')
        return None

    df['_f'] = pd.to_datetime(df[dc_match], errors='coerce')
    df = df.dropna(subset=['_f']).set_index('_f').sort_index()
    serie = pd.to_numeric(df[vc_match], errors='coerce').dropna()
    serie.name = label

    if fecha_inicio: serie = serie[serie.index >= pd.Timestamp(fecha_inicio)]
    if fecha_fin:    serie = serie[serie.index <= pd.Timestamp(fecha_fin)]

    return _resample_mean(serie, frecuencia).dropna()


# ── 7. Construcción de variable desde fila CSV ────────────────────────────────
def construir_variable(row: pd.Series, frecuencia: str,
                        fecha_inicio: str, fecha_fin: str):
    """
    Interpreta una fila del CSV de análisis y devuelve (serie, etiqueta).

    Flujo:
      1. Carga Serie A desde CATALOG (por su nombre original, SIN deflactor).
      2. Aplica Deflactor A si corresponde.
      3. Carga Serie B si corresponde y aplica Deflactor B.
      4. Opera A y B según "Operación".
      5. Aplica Log y/o Diferencia sobre el resultado final.
    """
    nombre_a  = str(row.get('Serie A',   '') or '').strip()
    defl_a    = str(row.get('Deflactor A','') or '').strip()
    nombre_b  = str(row.get('Serie B',   '') or '').strip()
    defl_b    = str(row.get('Deflactor B','') or '').strip()
    operacion = str(row.get('Operación', 'Solo A') or 'Solo A').strip()
    log_flag  = str(row.get('Log', 'No') or 'No').strip().lower() not in ('no', '0', 'false', '')
    n_diff    = int(float(str(row.get('Dif.', 0) or 0) or 0))

    # Etiqueta legible: solo incluye deflactor y serie B si realmente se aplican
    _b_activo  = operacion != 'Solo A' and nombre_b not in ('', '—', '-', 'nan')
    _da_activo = defl_a not in ('', '—', '-', 'nan')
    _db_activo = _b_activo and defl_b not in ('', '—', '-', 'nan')

    etiqueta = nombre_a
    if _da_activo:    etiqueta += f'/{defl_a}'
    if _b_activo:     etiqueta += f' {operacion} {nombre_b}'
    if _db_activo:    etiqueta += f'/{defl_b}'
    if log_flag:      etiqueta  = f'log({etiqueta})'
    if n_diff > 0:    etiqueta  = ('Δ' * n_diff) + etiqueta

    # 1. Cargar Serie A (siempre por nombre original, sin deflactor)
    sa = _cargar_serie_catalog(nombre_a, frecuencia, fecha_inicio, fecha_fin)
    if sa is None:
        return None, etiqueta

    # 2. Deflactar Serie A
    sa = _aplicar_deflactor(sa, defl_a, frecuencia)
    resultado = sa.copy()

    # 3. Cargar y deflactar Serie B, luego operar con A
    if _b_activo:
        sb = _cargar_serie_catalog(nombre_b, frecuencia, fecha_inicio, fecha_fin)
        if sb is not None:
            sb    = _aplicar_deflactor(sb, defl_b, frecuencia)
            sb_al = sb.reindex(sa.index, method='ffill')
            # Normalizar el símbolo menos Unicode (U+2212) → guion ASCII
            op = operacion.replace(' ', '').replace('−', '-')
            if op == 'A-B':                      resultado = sa - sb_al
            elif op in ('A/B', 'A÷B'):            resultado = sa / sb_al
            elif op in ('(A-B)/B', '(A-B)/B'):    resultado = (sa - sb_al) / sb_al
            elif op == 'A+B':                     resultado = sa + sb_al
            elif op in ('A*B', 'A×B'):             resultado = sa * sb_al
            elif op in ('SoloB', 'B'):             resultado = sb_al.copy()

    # 4. Transformaciones finales sobre el resultado de la operación
    if log_flag:
        resultado = np.log(resultado.replace(0, np.nan))
    for _ in range(n_diff):
        resultado = resultado.diff()

    resultado = resultado.dropna()
    resultado.name = etiqueta
    return resultado, etiqueta


# ── 8. Validación rápida del CATALOG vs CSV ───────────────────────────────────
def _validar_catalog_vs_csv(df_csv):
    print('\n── Validación CATALOG vs CSV ────────────────────────────────────────')
    seen = set()
    for _, row in df_csv.iterrows():
        for col in ['Serie A', 'Serie B']:
            nombre = str(row.get(col, '') or '').strip()
            if not nombre or nombre in ('', '—', '-', 'nan') or nombre in seen:
                continue
            seen.add(nombre)
            if nombre not in CATALOG_VIZ:
                print(f'  ❌ NO en CATALOG: "{nombre}"')
                continue
            meta      = CATALOG_VIZ[nombre]
            file_name = meta.get('file', '?')
            existe    = (VARS_DIR / file_name).exists() or (RAW_DIR / file_name).exists()
            print(f'  {"✅" if existe else "⚠️ "} {nombre[:55]:<55} → {file_name}'
                  f'{"" if existe else "  (¡FALTA ARCHIVO!)"}')
    print('────────────────────────────────────────────────────────────────────\n')


print('✅ Funciones auxiliares listas.')
print(f'   VARS_DIR: {VARS_DIR}')
print(f'   RAW_DIR:  {RAW_DIR}')


✅ CATALOG parseado: 197 series
✅ Funciones auxiliares listas.
   VARS_DIR: C:\Users\Usuario\Desktop\MARTIN\ECONOMICS\TRABAJO FINAL\Trabajo\data\Variables Finales
   RAW_DIR:  C:\Users\Usuario\Desktop\MARTIN\ECONOMICS\TRABAJO FINAL\Trabajo\data\raw


## 3. ══════════ CONFIGURACIÓN ══════════
> **Esta es la única celda que necesitás editar.**

In [3]:
# ── Archivo CSV ───────────────────────────────────────────────────────────────
# El visualizador descarga el archivo a la carpeta del navegador.
# Copiarlo a: notebooks/Variables Regresivas/
# Dejar None para que tome automáticamente el más reciente.
# Para un archivo específico usar solo el nombre: 'analisis_series_2026-05-06.csv'
CSV_ANALISIS = None   # None = el más reciente en notebooks/Variables Regresivas/

# ── Variable dependiente ──────────────────────────────────────────────────────
# Se elige en la Sección 4.5, luego de cargar y mostrar todas las series.

# ── Frecuencia ────────────────────────────────────────────────────────────────
# 'auto' = detecta la frecuencia más gruesa disponible entre todas las series
# Alternativas explícitas: 'D' 'W' 'M' 'Q' 'A'
FRECUENCIA = 'auto'

# ── Fechas ────────────────────────────────────────────────────────────────────
FECHA_INICIO = '2007-01-01'
FECHA_FIN    = '2024-12-31'

# ── Lags máximos ──────────────────────────────────────────────────────────────
MAX_LAGS_DEP = 4   # rezagos Y en ARDL
MAX_LAGS_IND = 4   # rezagos X en ARDL
MAX_LAGS_VAR = 8   # selección de rezagos en VAR/VECM

# ── Combinaciones ─────────────────────────────────────────────────────────────
# None = TODAS las combinaciones posibles de TODOS los tamaños
MAX_VARS_POR_MODELO = None   # None = sin límite (todas las variables)
MAX_COMBINACIONES   = None   # None = sin límite (todas las combinaciones)
SEMILLA             = 42

# ── Criterios de información ──────────────────────────────────────────────────
CRITERIOS_SEL = ['aic', 'bic', 'hqic']

# ── Nivel de significancia ────────────────────────────────────────────────────
NIVEL_SIG = 0.05

# Modelos a correr: ejecutar independientemente secciones 7 (ARDL), 8 (VAR), 9 (VECM)

print('✅ Configuración guardada.')

✅ Configuración guardada.


## 4. Carga y Construcción del Panel

In [4]:
random.seed(SEMILLA)
np.random.seed(SEMILLA)

# 1. Encontrar CSV ─────────────────────────────────────────────────────────────
_csv_dir = Path('Variables Regresivas')
if CSV_ANALISIS:
    _csv_path = _csv_dir / CSV_ANALISIS if not Path(CSV_ANALISIS).is_absolute() else Path(CSV_ANALISIS)
else:
    _candidates = sorted(glob.glob(str(_csv_dir / 'analisis_series_*.csv')))
    if not _candidates:
        raise FileNotFoundError(f'No encontré analisis_series_*.csv en {_csv_dir.resolve()}')
    _csv_path = Path(_candidates[-1])

print(f'📁 CSV: {_csv_path.name}')

# 2. Leer CSV ──────────────────────────────────────────────────────────────────
df_csv = pd.read_csv(_csv_path)
df_csv.columns = df_csv.columns.str.strip()
for _c in ['Deflactor A', 'Deflactor B', 'Serie B', 'Operación', 'Log']:
    if _c in df_csv.columns:
        df_csv[_c] = df_csv[_c].fillna('—').astype(str).str.strip()
if 'Dif.' in df_csv.columns:
    df_csv['Dif.'] = (pd.to_numeric(
        df_csv['Dif.'].astype(str).str.extract(r'(\d+)')[0], errors='coerce'
    ).fillna(0).astype(int))

print(f'   {len(df_csv)} variables en el CSV')
display(df_csv[['#','Serie A','Deflactor A','Serie B','Operación','Log','Dif.']].to_string())

# 3. Validar que todas las series del CSV estén en el CATALOG ─────────────────
_validar_catalog_vs_csv(df_csv)

# 4. Detectar frecuencia ───────────────────────────────────────────────────────
if FRECUENCIA == 'auto':
    FRECUENCIA_USADA = detectar_frecuencia(df_csv)
    print(f'🔍 Frecuencia auto-detectada: {FRECUENCIA_USADA}')
else:
    FRECUENCIA_USADA = FRECUENCIA
    print(f'📅 Frecuencia configurada: {FRECUENCIA_USADA}')

# 5. Construir todas las series (sin distinción Y/X) ───────────────────────────
print(f'\n⏳ Cargando series ({FECHA_INICIO} → {FECHA_FIN})...')
_series  = {}   # {etiqueta: pd.Series}
_eti_map = {}   # {índice CSV (0-based): etiqueta}

for _i, _row in df_csv.iterrows():
    try:
        _desde = str(_row.get('Desde', '') or '').strip()
        _hasta = str(_row.get('Hasta', '') or '').strip()
        _fi = _desde if _desde and _desde not in ('—', '-', 'nan') else FECHA_INICIO
        _ff = _hasta if _hasta and _hasta not in ('—', '-', 'nan') else FECHA_FIN
        _s, _eti = construir_variable(_row, FRECUENCIA_USADA, _fi, _ff)
        if _s is not None and len(_s) >= 20:
            _series[_eti] = _s
            _eti_map[_i]  = _eti
            print(f'  ✅ [{_i+1}] {_eti[:75]}  →  {len(_s)} obs')
        else:
            print(f'  ❌ [{_i+1}] {_row["Serie A"]}: no cargó o < 20 obs')
    except Exception as _e:
        print(f'  ❌ [{_i+1}] {_row["Serie A"]}: {_e}')

if not _series:
    raise RuntimeError('No se cargó ninguna serie.')

# 6. Panel unificado ───────────────────────────────────────────────────────────
df_panel = pd.concat(list(_series.values()), axis=1).dropna()
print(f'\n✅ Panel: {df_panel.shape[0]} obs × {df_panel.shape[1]} vars')
print(f'   Período: {df_panel.index.min().date()} → {df_panel.index.max().date()}')

# 7. Lista numerada para selección de variable dependiente ────────────────────
print('\n' + '═'*70)
print('  Series disponibles — elegí la variable dependiente en la Sección 4.5')
print('═'*70)
_LISTA_SERIES = list(df_panel.columns)
for _n, _col in enumerate(_LISTA_SERIES, 1):
    print(f'  [{_n}]  {_col}')
print('═'*70)

📁 CSV: analisis_series_2026-05-08.csv
   7 variables en el CSV


'   #                            Serie A Deflactor A              Serie B Operación Log  Dif.\n0  1               Reservas Brutas BCRA         PBI                    —    Solo A  No     1\n1  2                 ITCRM Multilateral           —                    —    Solo A  No     1\n2  3            Riesgo País ARG (EMBI+)           —  EMBI+ Latinoamérica     A − B  No     1\n3  4  Términos de Intercambio (nominal)           —                    —     A − B  No     1\n4  5                    Saldo Comercial         PBI                    —     A − B  No     1\n5  6       Brecha Cambiaria MEP/Oficial           —                    —     A − B  No     1\n6  7        VIX — CBOE Volatility Index           —                    —     A − B  No     1'


── Validación CATALOG vs CSV ────────────────────────────────────────
  ✅ Reservas Brutas BCRA                                    → 001_reservas_brutas.csv
  ✅ ITCRM Multilateral                                      → itcrm_diario.csv
  ✅ Riesgo País ARG (EMBI+)                                 → riesgo_pais_arg.csv
  ✅ EMBI+ Latinoamérica                                     → embi_latam.csv
  ✅ Términos de Intercambio (nominal)                       → ti_mensual.csv
  ✅ Saldo Comercial                                         → ica_intercambio_comercial_argentino_valores_mensuales.csv
  ✅ Brecha Cambiaria MEP/Oficial                            → brecha_cambiaria.csv
  ✅ VIX — CBOE Volatility Index                             → vix.csv
────────────────────────────────────────────────────────────────────

🔍 Frecuencia auto-detectada: M

⏳ Cargando series (2007-01-01 → 2024-12-31)...
  ✅ [1] ΔReservas Brutas BCRA/PBI  →  215 obs
  ✅ [2] ΔITCRM Multilateral  →  215 obs
  ✅ [3] ΔRiesgo País

## 4.5 ─── Seleccionar variable dependiente ───
> Corré la celda de abajo: imprime la lista y selecciona automáticamente riesgo país.  
> Si querés otra variable, cambiá el número en `VARIABLE_DEPENDIENTE` y volvé a correrla.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  ELEGÍ LA VARIABLE DEPENDIENTE — cambiá solo el número de abajo
#  y volvé a correr esta celda
# ══════════════════════════════════════════════════════════════════════════════

# Auto-detecta riesgo país; si no lo encuentra usa 1
_KEYWORDS_DEP = ['embi', 'riesgo', 'spread', 'cds', 'country risk']
_auto_idx = next(
    (i + 1 for i, c in enumerate(_LISTA_SERIES)
     if any(kw in c.lower() for kw in _KEYWORDS_DEP)),
    1
)

VARIABLE_DEPENDIENTE = _auto_idx   # ← CAMBIÁ ESTE NÚMERO SI QUERÉS OTRA VARIABLE

# ── Lista disponible ──────────────────────────────────────────────────────────
print('═' * 65)
for _i, _c in enumerate(_LISTA_SERIES, 1):
    marca = '  ◄ seleccionada' if _i == VARIABLE_DEPENDIENTE else ''
    print(f'  [{_i:>2}]  {_c}{marca}')
print('═' * 65)

# ── Aplicar selección ─────────────────────────────────────────────────────────
if not (1 <= VARIABLE_DEPENDIENTE <= len(_LISTA_SERIES)):
    raise ValueError(f'VARIABLE_DEPENDIENTE={VARIABLE_DEPENDIENTE} fuera de rango (1–{len(_LISTA_SERIES)})')

ETI_DEP  = _LISTA_SERIES[VARIABLE_DEPENDIENTE - 1]
ETI_EXPL = [c for c in _LISTA_SERIES if c != ETI_DEP]

_all_cols = [ETI_DEP] + ETI_EXPL
COL_SAFE  = {c: f'v{i:02d}' for i, c in enumerate(_all_cols)}
COL_LABEL = {v: k for k, v in COL_SAFE.items()}
df_safe   = df_panel[_all_cols].rename(columns=COL_SAFE)
Y_SAFE    = COL_SAFE[ETI_DEP]
X_SAFE    = [COL_SAFE[e] for e in ETI_EXPL]

print(f'\n✅ Y = {ETI_DEP}')
print(f'   {len(ETI_EXPL)} variables explicativas')


## 5. Estadísticos Descriptivos

In [ ]:
display(resumen_estadistico(df_panel).round(4))

n_series = len(df_panel.columns)
fig, axes = plt.subplots(n_series, 1, figsize=(12, 3 * n_series), sharex=True, squeeze=False)

for i, col in enumerate(df_panel.columns):
    color = '#d62728' if col == ETI_DEP else '#1f77b4'
    axes[i][0].plot(df_panel.index, df_panel[col], color=color, linewidth=1.2)
    axes[i][0].set_title(col, fontsize=9)

fig.suptitle('Series del panel  (rojo = variable dependiente)', fontsize=11)
plt.tight_layout()
plt.show()


## 6. Tests de Estacionariedad

In [ ]:
# ── Tests ADF + KPSS ──────────────────────────────────────────────────────────
print('═'*70)
print('6.  TESTS DE ESTACIONARIEDAD (ADF + KPSS)')
print('═'*70)
_est_rows = []
for _col in df_panel.columns:
    _r = test_estacionariedad(df_panel[_col], _col)
    _est_rows.append(_r)
df_est = pd.DataFrame(_est_rows).set_index('Variable')
display(df_est[['ADF_stat','ADF_pvalue','ADF_estacionaria',
                'KPSS_stat','KPSS_pvalue','KPSS_estacionaria','Conclusión']])

# ── Identificar series no estacionarias ───────────────────────────────────────
_non_stat_labels = set(
    df_est.index[df_est['Conclusión'].isin(['No Estacionaria', 'Conflicto'])]
)

# Diccionario safe_name -> label de las no estacionarias (excluyendo dep. variable)
VARS_NO_ESTACIONARIAS = {
    _s: COL_LABEL.get(_s, _s)
    for _s in X_SAFE
    if COL_LABEL.get(_s, _s) in _non_stat_labels
}

# Variables explicativas disponibles para estimación (solo estacionarias)
X_SAFE_EST = [v for v in X_SAFE if v not in VARS_NO_ESTACIONARIAS]

# ── Aviso al usuario ───────────────────────────────────────────────────────────
if VARS_NO_ESTACIONARIAS:
    _dep_non_stat = ETI_DEP in _non_stat_labels
    print('\n' + '='*70)
    print('  ADVERTENCIA - VARIABLES NO ESTACIONARIAS DETECTADAS')
    print('='*70)
    if _dep_non_stat:
        print(f'\n  La variable dependiente "{ETI_DEP}" no es estacionaria.')
        print('  Considere transformarla (diferencias, log) antes de estimar.')
    print(f'\n  Variables explicativas no estacionarias ({len(VARS_NO_ESTACIONARIAS)}):')
    for _s, _l in VARS_NO_ESTACIONARIAS.items():
        print(f'    - {_l}')
    print()
    print('  Estas variables seran DESCARTADAS automaticamente de las regresiones.')
    print('  Para incluirlas, aplique una transformacion (delta, log, log-delta, etc.)')
    print('  y vuelva a correr desde la Seccion 4 con los datos transformados.')
    print('='*70)
else:
    print('\nTodas las variables son estacionarias. Puede proceder con las estimaciones.')

print(f'\nVariables para estimacion: {len(X_SAFE_EST)} de {len(X_SAFE)} explicativas disponibles.')
if VARS_NO_ESTACIONARIAS:
    print(f'Descartadas: {", ".join(VARS_NO_ESTACIONARIAS.values())}')
print('\nTests de estacionariedad completados.')


## 7. Modelos ARDL

### 7.1 Tests y Transformaciones Previos — Prewhitening Box-Jenkins

> El análisis Box-Jenkins determina los lags óptimos de cada regresor para el ARDL. Se ajusta un ARMA a cada X, se aplica el filtro a Y (*prewhitening*) y se calcula la CCF para identificar los lags significativos. Variables no estacionarias (Sección 6) son excluidas.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 6.4  Análisis BJ: prewhitening correcto (X filtra a Y) + propuesta de lags
# ═══════════════════════════════════════════════════════════════════════════
try:
    from pmdarima import auto_arima as _auto_arima_bj
except ImportError:
    import subprocess as _sp_bj, sys as _sys_bj
    _sp_bj.check_call([_sys_bj.executable, '-m', 'pip', 'install', 'pmdarima', '-q'])
    from pmdarima import auto_arima as _auto_arima_bj

from statsmodels.tsa.stattools import ccf as _ccf_fn_bj
import statsmodels.api as _sm_bj

BJ_MAX_LAGS_CCF = 12
BJ_MAX_AR       = 4
BJ_Z_CI         = 1.96

_y_bj = df_safe[Y_SAFE].dropna()

_BJ_PROPOSED_LAGS  = {}
_bj_ar_y_estimates = []
_bj_ccf_store      = {}

for _xvar_bj in X_SAFE_EST:
    _xlabel_bj = COL_LABEL[_xvar_bj]
    _xs        = df_safe[_xvar_bj].dropna()
    _df_xy_bj  = pd.concat([_y_bj.rename('y'), _xs.rename('x')], axis=1).dropna()
    if len(_df_xy_bj) < 30:
        _BJ_PROPOSED_LAGS[_xvar_bj] = [0, 1]
        continue

    try:
        # ── Paso 1: ajustar ARMA a X ─────────────────────────────────────────
        _fit_x_bj = _auto_arima_bj(
            _df_xy_bj['x'].values, information_criterion='aic',
            stepwise=True, seasonal=False, d=0,
            max_p=BJ_MAX_AR, max_q=BJ_MAX_AR,
            suppress_warnings=True, error_action='ignore',
        )
        _p_x_bj, _, _q_x_bj = _fit_x_bj.order
        _x_resid = _fit_x_bj.resid()   # X preblanqueada

        # ── Paso 2: aplicar el filtro AR de X sobre Y (prewhitening correcto) ─
        #   y_filt[t] = y[t] − φ₁·y[t-1] − … − φₚ·y[t-p]
        _ar_x = _fit_x_bj.arparams()   # coefs AR del modelo de X
        _y_arr = _df_xy_bj['y'].values
        _p_ar  = len(_ar_x)
        if _p_ar > 0:
            _y_filt = np.array([
                _y_arr[t] - np.dot(_ar_x, _y_arr[t - _p_ar:t][::-1])
                for t in range(_p_ar, len(_y_arr))
            ])
        else:
            _y_filt = _y_arr.copy()

        # ── Paso 3: alinear longitudes y calcular CCF ─────────────────────────
        _n_min   = min(len(_x_resid), len(_y_filt))
        _xw      = _x_resid[-_n_min:]
        _yw      = _y_filt[-_n_min:]
        _ci_bj   = BJ_Z_CI / np.sqrt(_n_min)
        _ccf_v_bj = _ccf_fn_bj(_xw, _yw, nlags=BJ_MAX_LAGS_CCF, alpha=None)

        _sig_bj  = [int(l) for l in range(len(_ccf_v_bj)) if abs(_ccf_v_bj[l]) > _ci_bj]
        _prop_bj = sorted({l for l in _sig_bj if 0 <= l <= MAX_LAGS_IND})
        if not _prop_bj:
            _prop_bj = [0, 1]

        # ── Paso 4: AR(Y) estimado sobre residuos de OLS (para ARDL) ─────────
        _df_ols_bj = _df_xy_bj.copy()
        for _lag_bj in _prop_bj:
            _df_ols_bj[f'xL{_lag_bj}'] = _df_ols_bj['x'].shift(_lag_bj)
        _df_ols_bj = _df_ols_bj.dropna()
        _Xm_bj = _sm_bj.add_constant(_df_ols_bj[[f'xL{l}' for l in _prop_bj]])
        _resid_ols_bj = _sm_bj.OLS(_df_ols_bj['y'], _Xm_bj).fit().resid
        _fit_r_bj = _auto_arima_bj(
            _resid_ols_bj.values, information_criterion='aic',
            stepwise=True, seasonal=False, d=0,
            max_p=BJ_MAX_AR, max_q=BJ_MAX_AR,
            suppress_warnings=True, error_action='ignore',
        )
        _p_r_bj, _, _ = _fit_r_bj.order
        _bj_ar_y_estimates.append(max(_p_r_bj, 1))

        _BJ_PROPOSED_LAGS[_xvar_bj] = _prop_bj
        _bj_ccf_store[_xvar_bj] = {
            'label'  : _xlabel_bj,
            'arma_x' : (_p_x_bj, _q_x_bj),
            'ccf'    : _ccf_v_bj,
            'ci'     : _ci_bj,
            'prop'   : _prop_bj,
        }
        print(f'  {_xlabel_bj[:55]:<55}  ARMA_X=({_p_x_bj},{_q_x_bj})  lags={_prop_bj}')

    except Exception as _e_bj:
        _BJ_PROPOSED_LAGS[_xvar_bj] = [0, 1]
        print(f'  WARN {COL_LABEL[_xvar_bj][:40]}: {_e_bj}')

_BJ_AR_Y = int(round(np.median(_bj_ar_y_estimates))) if _bj_ar_y_estimates else 1
print(f'\n  AR(Y) global propuesto: {_BJ_AR_Y}')

# ── Gráfico CCF ───────────────────────────────────────────────────────────────
if _bj_ccf_store:
    _nv = len(_bj_ccf_store)
    _nc = 2
    _nr = (_nv + _nc - 1) // _nc
    fig, axes = plt.subplots(_nr, _nc, figsize=(10 * _nc, 4 * _nr), squeeze=False)

    for _idx, (_xvar, _res) in enumerate(_bj_ccf_store.items()):
        ax      = axes[_idx // _nc][_idx % _nc]
        _cv     = _res['ccf']
        _ci_v   = _res['ci']
        _prop_v = set(_res['prop'])
        _lgs    = list(range(len(_cv)))
        _clrs   = ['#d62728' if l in _prop_v
                   else '#1565C0' if abs(_cv[l]) > _ci_v
                   else '#BDBDBD' for l in _lgs]
        ax.bar(_lgs, _cv, color=_clrs)
        ax.axhline( _ci_v, color='red',   linestyle='--', linewidth=1)
        ax.axhline(-_ci_v, color='red',   linestyle='--', linewidth=1)
        ax.axhline(0,       color='black', linewidth=0.5)
        ax.set_title(
            f"{_res['label'][:40]}  ARMA_X=({_res['arma_x'][0]},{_res['arma_x'][1]})",
            fontsize=8)
        ax.set_xlabel(f"Lag  |  propuestos={sorted(_prop_v)}")
        ax.set_ylabel('CCF  (Y filtrada por X)')

    for _idx in range(_nv, _nr * _nc):
        axes[_idx // _nc][_idx % _nc].set_visible(False)

    fig.suptitle(
        'CCF: X blanqueada vs Y filtrada por AR(X)  '
        '(rojo=propuesto, azul=sig, gris=no sig)', fontsize=10)
    plt.tight_layout()
    plt.show()

print('OK  _BJ_PROPOSED_LAGS y _BJ_AR_Y listos.')


### 7.1.1 Funciones Auxiliares de Estimación

> Estas funciones son usadas por las secciones 7, 8 y 9. Ejecutar esta celda antes de correr cualquier estimación.

In [ ]:
def _hqic(llf, k, n):
    return -2 * llf + 2 * k * np.log(np.log(max(n, 3)))


def extraer_ardl(df_m, y_var, x_vars, max_lags_dep, max_lags_ind, criterio):
    from statsmodels.tsa.ardl import ARDL as _ARDL_direct

    data = df_m[[y_var] + x_vars].dropna()
    y    = data[y_var]
    Xd   = data[x_vars]

    _bj_lags_ok = '_BJ_PROPOSED_LAGS' in globals()
    if _bj_lags_ok:
        try:
            _bj_lp  = globals()['_BJ_PROPOSED_LAGS']
            _bj_ary = int(globals().get('_BJ_AR_Y', 1))
            _order_dict = {x: max(_bj_lp.get(x, [0, 1])) for x in x_vars}
            res = _ARDL_direct(y, lags=_bj_ary, exog=Xd,
                               order=_order_dict, trend='c').fit()
        except Exception:
            _bj_lags_ok = False

    if not _bj_lags_ok:
        from statsmodels.tsa.ardl import ardl_select_order
        sel = ardl_select_order(y, max_lags_dep, Xd, max_lags_ind,
                                ic=criterio, trend='c')
        res = sel.model.fit()

    n   = res.nobs
    k   = len(res.params)
    aic = float(res.aic)
    bic = float(res.bic)
    hq  = float(getattr(res, 'hqic', _hqic(res.llf, k, n)))
    r2  = float(getattr(res, 'rsquared', np.nan))

    coefs = {}
    for x in x_vars:
        px = [p for p in res.params.index if p.startswith(f'{x}.L') or p == x]
        if not px:
            px = [p for p in res.params.index
                  if p == x or (p.startswith('L') and p.split('.', 1)[-1] == x)]
        if px:
            coefs[x] = {'coef': float(res.params[px].sum()),
                        'pval': float(res.pvalues[px].min())}
        else:
            coefs[x] = {'coef': np.nan, 'pval': np.nan}

    return {'aic': aic, 'bic': bic, 'hq': hq, 'r2': r2,
            'coefs': coefs, 'nobs': int(n)}


def extraer_var(df_m, y_var, x_vars, todas_vars, max_lags, criterio):
    from statsmodels.tsa.api import VAR

    data    = df_m[todas_vars].dropna()
    model   = VAR(data)
    lag_sel = model.select_order(maxlags=max_lags)
    k_ar    = max(1, lag_sel.selected_orders.get(criterio, 1))
    res     = model.fit(k_ar)

    aic = float(res.aic)
    bic = float(res.bic)
    hq  = float(res.hqic)

    # R² para la ecuación Y
    _y_pos   = list(todas_vars).index(y_var)
    _y_resid = res.resid[:, _y_pos]
    _y_act   = data[y_var].values[-len(_y_resid):]
    _ss_res  = float(np.sum(_y_resid ** 2))
    _ss_tot  = float(np.sum((_y_act - _y_act.mean()) ** 2))
    r2 = float(1 - _ss_res / _ss_tot) if _ss_tot > 0 else np.nan

    params_df = res.params
    pvals_df  = res.pvalues
    k_endog   = len(todas_vars)
    var_pos   = {v: i for i, v in enumerate(todas_vars)}

    coefs = {}
    if y_var in params_df.columns:
        yp = params_df[y_var]
        yv = pvals_df[y_var]
        for x in x_vars:
            xpos = var_pos.get(x)
            if xpos is None:
                coefs[x] = {'coef': np.nan, 'pval': np.nan}
                continue
            idx = [lag * k_endog + xpos for lag in range(k_ar)
                   if lag * k_endog + xpos < len(yp)]
            coefs[x] = ({'coef': float(yp.iloc[idx].sum()),
                         'pval': float(yv.iloc[idx].min())}
                        if idx else {'coef': np.nan, 'pval': np.nan})
    else:
        for x in x_vars:
            coefs[x] = {'coef': np.nan, 'pval': np.nan}

    return {'aic': aic, 'bic': bic, 'hq': hq, 'r2': r2,
            'coefs': coefs, 'k_ar': k_ar, 'nobs': int(res.nobs)}


def extraer_vecm(df_m, y_var, x_vars, todas_vars, k_ar_diff, det_order):
    from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
    from scipy.stats import t as t_dist

    data = df_m[todas_vars].dropna()
    joh  = coint_johansen(data, det_order=det_order, k_ar_diff=k_ar_diff)
    n_ci = int(sum(joh.lr1 > joh.cvt[:, 1]))
    if n_ci == 0:
        return None

    res = VECM(data, k_ar_diff=k_ar_diff, coint_rank=n_ci,
               deterministic='ci').fit()

    aic = float(res.aic)
    bic = float(res.bic)
    hq  = float(getattr(res, 'hqic',
                         _hqic(res.llf if hasattr(res, 'llf') else
                               -res.bic * res.nobs / 2,
                               len(todas_vars) * n_ci, res.nobs)))

    # R² para la ecuación Y (en diferencias)
    all_list = list(todas_vars)
    y_idx    = all_list.index(y_var)
    _y_resid_v = res.resid[:, y_idx]
    _y_act_v   = data[y_var].diff().dropna().values[-len(_y_resid_v):]
    _ss_res_v  = float(np.sum(_y_resid_v ** 2))
    _ss_tot_v  = float(np.sum((_y_act_v - _y_act_v.mean()) ** 2))
    r2 = float(1 - _ss_res_v / _ss_tot_v) if _ss_tot_v > 0 else np.nan

    k_vars   = len(all_list)
    gamma    = res.gamma
    gamma_se = getattr(res, 'gamma_se', None)
    n_short  = k_ar_diff - 1

    coefs = {}
    for x in x_vars:
        if x not in all_list:
            coefs[x] = {'coef': np.nan, 'pval': np.nan}
            continue
        xi   = all_list.index(x)
        cols = [l * k_vars + xi for l in range(n_short)
                if l * k_vars + xi < gamma.shape[1]]
        if cols:
            c  = float(gamma[y_idx, cols].sum())
            se = (float(np.sqrt(np.sum(gamma_se[y_idx, cols] ** 2)))
                  if gamma_se is not None else None)
            pv = (float(t_dist.sf(abs(c / (se + 1e-12)),
                                  df=max(1, res.nobs - k_vars)) * 2)
                  if se is not None else np.nan)
        else:
            c = pv = np.nan
        coefs[x] = {'coef': c, 'pval': pv}

    return {'aic': aic, 'bic': bic, 'hq': hq, 'r2': r2,
            'coefs': coefs, 'n_coint': n_ci, 'nobs': int(res.nobs)}


print('Funciones de extracción listas (con R²).')


### 7.2 Estimaciones — ARDL

> Estimación iterativa de modelos ARDL con todas las combinaciones posibles de `X_SAFE_EST`. Los lags se determinan por el prewhitening BJ (sección 7.1). Los resultados se almacenan en `df_res_ardl`.

In [ ]:
# ── Diagnóstico opcional: probar VAR/VECM en una combinación ────────────────
# Ejecutar solo para depurar errores en las estimaciones.
# Requiere que _combos_ardl esté definido (ejecutar 7.2 primero).
import traceback as _tb
try:
    _diag_combo = list(_combos_ardl[0])
    _diag_tv    = [Y_SAFE] + _diag_combo
    _diag_df    = df_safe[_diag_tv].dropna()
    print(f'Combo de prueba: {[COL_LABEL[v] for v in _diag_combo]}')
    print(f'N obs: {len(_diag_df)} | Frecuencia: {_diag_df.index.freq}')
except Exception as _e_diag:
    print(f'Diagnóstico no disponible: {_e_diag}')


In [ ]:
# ── Preparación de combinaciones ARDL ────────────────────────────────────────
_n_max_ardl  = len(X_SAFE_EST) if MAX_VARS_POR_MODELO is None else MAX_VARS_POR_MODELO
_combos_ardl = []
for _r in range(1, _n_max_ardl + 1):
    _combos_ardl.extend(list(combinations(X_SAFE_EST, _r)))

if MAX_COMBINACIONES is not None and len(_combos_ardl) > MAX_COMBINACIONES:
    random.seed(SEMILLA)
    _combos_ardl = random.sample(_combos_ardl, MAX_COMBINACIONES)
    print(f'ARDL: muestreadas {MAX_COMBINACIONES} de {len(_combos_ardl)} combinaciones')

print(f'Combinaciones ARDL : {len(_combos_ardl)}')
print(f'Criterios          : {CRITERIOS_SEL}')
print(f'Estimaciones esp.  : {len(_combos_ardl) * len(CRITERIOS_SEL)}')
print()

_RESULTADOS_ARDL = []
_t0_ardl = datetime.now()
_err_ardl = 0
_ok_ardl  = 0
_PRINT_INT_ARDL = max(1, len(_combos_ardl) // 5)

for _criterio in CRITERIOS_SEL:
    for _ci, _combo in enumerate(_combos_ardl):
        _xv = list(_combo)
        _tv = [Y_SAFE] + _xv
        _df = df_safe[_tv].dropna()
        if len(_df) < 30:
            continue
        try:
            _r = extraer_ardl(_df, Y_SAFE, _xv, MAX_LAGS_DEP, MAX_LAGS_IND, _criterio)
            _row = {
                'combo_id'  : f'{_ci}_{_criterio}',
                'criterio'  : _criterio,
                'modelo'    : 'ARDL',
                'n_vars'    : len(_xv),
                'vars_safe' : '|'.join(_xv),
                'vars_label': '|'.join([COL_LABEL[v] for v in _xv]),
                'aic'       : _r['aic'],
                'bic'       : _r['bic'],
                'hq'        : _r['hq'],
                'r2'        : _r['r2'],
                'nobs'      : _r['nobs'],
            }
            for _x in _xv:
                _row[f'coef_{_x}'] = _r['coefs'][_x]['coef']
                _row[f'pval_{_x}'] = _r['coefs'][_x]['pval']
            _RESULTADOS_ARDL.append(_row)
            _ok_ardl += 1
        except Exception:
            _err_ardl += 1

        if (_ci + 1) % _PRINT_INT_ARDL == 0 or _ci == len(_combos_ardl) - 1:
            _el = (datetime.now() - _t0_ardl).seconds
            print(f'  [{_criterio.upper()}] combo {_ci+1:>3}/{len(_combos_ardl)} | '
                  f'ok={_ok_ardl}  err={_err_ardl}  | {_el}s')

df_res_ardl = pd.DataFrame(_RESULTADOS_ARDL)
_tasa_a = f'{_ok_ardl/(_ok_ardl+_err_ardl)*100:.1f}%' if (_ok_ardl+_err_ardl) > 0 else 'n/a'
print(f'\n{"="*60}')
print(f'ARDL completado: {_ok_ardl} modelos | {_err_ardl} errores | tasa={_tasa_a}')
print(f'{"="*60}')


### 7.3 Control — ARDL

> Filtrado de modelos ARDL según criterios de calidad. Se configura R² mínimo, significancia mínima de al menos una variable, y otros criterios. Los modelos que pasan se almacenan en `df_res_ardl_ok`.

In [ ]:
# ── Control de calidad ARDL ──────────────────────────────────────────────────
print('='*70)
print('7.3  CONTROL DE CALIDAD — MODELOS ARDL')
print('='*70)

if df_res_ardl.empty:
    print('Sin resultados ARDL para controlar.')
    df_res_ardl_ok = pd.DataFrame()
else:
    # ── Parámetros de control (ajustar según necesidad) ───────────────────────
    CTRL_ARDL_MIN_R2   = 0.0   # R² mínimo aceptable (0 = sin filtro)
    CTRL_ARDL_MAX_PVAL = 1.0   # al menos 1 var debe tener p < este valor (1 = sin filtro)

    _ardl_flags = {row['combo_id']: [] for _, row in df_res_ardl.iterrows()}

    # ── Filtro R² ─────────────────────────────────────────────────────────────
    if CTRL_ARDL_MIN_R2 > 0:
        _low_r2 = df_res_ardl[df_res_ardl['r2'] < CTRL_ARDL_MIN_R2]['combo_id'].tolist()
        for _cid in _low_r2:
            _ardl_flags[_cid].append(f'R2_bajo')
        print(f'  R2 < {CTRL_ARDL_MIN_R2}: {len(_low_r2)} modelos marcados')

    # ── Al menos una variable significativa ───────────────────────────────────
    _pval_cols = [c for c in df_res_ardl.columns if c.startswith('pval_')]
    if _pval_cols and CTRL_ARDL_MAX_PVAL < 1.0:
        _min_pval = df_res_ardl[_pval_cols].min(axis=1)
        _no_sig   = df_res_ardl[_min_pval > CTRL_ARDL_MAX_PVAL]['combo_id'].tolist()
        for _cid in _no_sig:
            _ardl_flags[_cid].append('ninguna_sig')
        print(f'  Ninguna var significativa (p>{CTRL_ARDL_MAX_PVAL}): {len(_no_sig)} modelos')

    # ── Resumen ───────────────────────────────────────────────────────────────
    _n_flagged = sum(1 for v in _ardl_flags.values() if v)
    _n_total   = len(df_res_ardl)
    print(f'\n  Total modelos ARDL   : {_n_total}')
    print(f'  Modelos con flags    : {_n_flagged}')
    print(f'  Modelos sin flags    : {_n_total - _n_flagged}  (pasan control)')

    df_res_ardl['ctrl_flags'] = df_res_ardl['combo_id'].map(
        lambda cid: '|'.join(_ardl_flags.get(cid, [])) or 'OK'
    )
    df_res_ardl['ctrl_ok'] = df_res_ardl['ctrl_flags'] == 'OK'
    df_res_ardl_ok = df_res_ardl[df_res_ardl['ctrl_ok']].copy()
    print(f'\n  df_res_ardl_ok: {len(df_res_ardl_ok)} modelos listos para analisis comparativo.')
    print('\nControl ARDL completado.')


## 8. Modelos VAR

### 8.1 Tests y Transformaciones Previos

> Selección del orden óptimo del VAR mediante criterios de información. Se evalúa sobre una muestra de combinaciones para estimar el rango de lags apropiado.

In [ ]:
# ── 8.1 Selección del orden VAR ──────────────────────────────────────────────
from statsmodels.tsa.api import VAR as _VAR_pretest
from itertools import combinations as _comb_pt

print('='*70)
print('8.1  SELECCION DE ORDEN VAR (criterios de informacion)')
print('='*70)

_var_order_rows = []
_sample_size    = min(10, len(list(_comb_pt(X_SAFE_EST, min(3, len(X_SAFE_EST))))))

for _ci, _xv_s in enumerate(_comb_pt(X_SAFE_EST, min(3, len(X_SAFE_EST)))):
    if _ci >= _sample_size:
        break
    _tv_s = [Y_SAFE] + list(_xv_s)
    _df_s = df_safe[_tv_s].dropna()
    if len(_df_s) < 30:
        continue
    try:
        _sel = _VAR_pretest(_df_s).select_order(maxlags=MAX_LAGS_VAR)
        _var_order_rows.append({
            'vars'   : ' | '.join([COL_LABEL.get(v, v)[:25] for v in _xv_s]),
            'AIC'    : _sel.selected_orders.get('aic',  None),
            'BIC'    : _sel.selected_orders.get('bic',  None),
            'HQIC'   : _sel.selected_orders.get('hqic', None),
            'n_obs'  : len(_df_s),
        })
    except Exception:
        pass

if _var_order_rows:
    _df_var_ord = pd.DataFrame(_var_order_rows)
    print(f'\n  Orden VAR sugerido (muestra de {len(_var_order_rows)} combinaciones):')
    display(_df_var_ord)
    _med_aic = int(_df_var_ord['AIC'].dropna().median()) if not _df_var_ord['AIC'].isna().all() else 1
    print(f'\n  Lag mediano sugerido por AIC : {_med_aic}')
    print(f'  Maximo de lags configurado   : {MAX_LAGS_VAR}')
else:
    print('  No se pudo calcular el orden VAR.')

print('\nTests previos VAR completados.')


### 8.2 Estimaciones — VAR

> Estimación iterativa de modelos VAR con todas las combinaciones de `X_SAFE_EST`. El orden de cada VAR se selecciona automáticamente por AIC. Los resultados se almacenan en `df_res_var`.

In [ ]:
# ── Preparación de combinaciones VAR ─────────────────────────────────────────
_n_max_var  = len(X_SAFE_EST) if MAX_VARS_POR_MODELO is None else MAX_VARS_POR_MODELO
_combos_var = []
for _r in range(1, _n_max_var + 1):
    _combos_var.extend(list(combinations(X_SAFE_EST, _r)))

if MAX_COMBINACIONES is not None and len(_combos_var) > MAX_COMBINACIONES:
    random.seed(SEMILLA)
    _combos_var = random.sample(_combos_var, MAX_COMBINACIONES)
    print(f'VAR: muestreadas {MAX_COMBINACIONES} de {len(_combos_var)} combinaciones')

print(f'Combinaciones VAR  : {len(_combos_var)}')
print(f'Criterios          : {CRITERIOS_SEL}')
print(f'Estimaciones esp.  : {len(_combos_var) * len(CRITERIOS_SEL)}')
print()

_RESULTADOS_VAR = []
_t0_var  = datetime.now()
_err_var = 0
_ok_var  = 0
_PRINT_INT_VAR = max(1, len(_combos_var) // 5)

for _criterio in CRITERIOS_SEL:
    for _ci, _combo in enumerate(_combos_var):
        _xv = list(_combo)
        _tv = [Y_SAFE] + _xv
        _df = df_safe[_tv].dropna()
        if len(_df) < 30:
            continue
        try:
            _r = extraer_var(_df, Y_SAFE, _xv, _tv, MAX_LAGS_VAR, _criterio)
            _row = {
                'combo_id'  : f'{_ci}_{_criterio}',
                'criterio'  : _criterio,
                'modelo'    : 'VAR',
                'n_vars'    : len(_xv),
                'vars_safe' : '|'.join(_xv),
                'vars_label': '|'.join([COL_LABEL[v] for v in _xv]),
                'aic'       : _r['aic'],
                'bic'       : _r['bic'],
                'hq'        : _r['hq'],
                'r2'        : _r['r2'],
                'nobs'      : _r['nobs'],
            }
            for _x in _xv:
                _row[f'coef_{_x}'] = _r['coefs'][_x]['coef']
                _row[f'pval_{_x}'] = _r['coefs'][_x]['pval']
            _RESULTADOS_VAR.append(_row)
            _ok_var += 1
        except Exception:
            _err_var += 1

        if (_ci + 1) % _PRINT_INT_VAR == 0 or _ci == len(_combos_var) - 1:
            _el = (datetime.now() - _t0_var).seconds
            print(f'  [{_criterio.upper()}] combo {_ci+1:>3}/{len(_combos_var)} | '
                  f'ok={_ok_var}  err={_err_var}  | {_el}s')

df_res_var = pd.DataFrame(_RESULTADOS_VAR)
_tasa_v = f'{_ok_var/(_ok_var+_err_var)*100:.1f}%' if (_ok_var+_err_var) > 0 else 'n/a'
print(f'\n{"="*60}')
print(f'VAR completado: {_ok_var} modelos | {_err_var} errores | tasa={_tasa_v}')
print(f'{"="*60}')


### 8.3 Control — VAR

> Filtrado de modelos VAR. Se verifican: R² mínimo, estabilidad del sistema (raíces del polinomio característico < 1 en módulo), y significancia de variables. Los modelos aprobados se almacenan en `df_res_var_ok`.

In [ ]:
# ── Control de calidad VAR ───────────────────────────────────────────────────
from statsmodels.tsa.api import VAR as _VAR_ctrl

print('='*70)
print('8.3  CONTROL DE CALIDAD — MODELOS VAR')
print('='*70)

if df_res_var.empty:
    print('Sin resultados VAR para controlar.')
    df_res_var_ok = pd.DataFrame()
else:
    CTRL_VAR_MIN_R2 = 0.0   # R² mínimo (0 = sin filtro)
    CTRL_VAR_ESTAB  = True  # verificar estabilidad del mejor modelo

    _var_flags = {row['combo_id']: [] for _, row in df_res_var.iterrows()}

    if CTRL_VAR_MIN_R2 > 0:
        _low_r2v = df_res_var[df_res_var['r2'] < CTRL_VAR_MIN_R2]['combo_id'].tolist()
        for _cid in _low_r2v:
            _var_flags[_cid].append('R2_bajo')
        print(f'  R2 < {CTRL_VAR_MIN_R2}: {len(_low_r2v)} modelos marcados')

    # Verificar estabilidad del mejor modelo
    if CTRL_VAR_ESTAB and not df_res_var.empty:
        _best_vc   = df_res_var.loc[df_res_var['aic'].idxmin()]
        _xv_vc     = [v for v in _best_vc['vars_safe'].split('|') if v]
        _tv_vc     = [Y_SAFE] + _xv_vc
        try:
            _df_vc  = df_safe[_tv_vc].dropna()
            _ls_vc  = _VAR_ctrl(_df_vc).select_order(maxlags=MAX_LAGS_VAR)
            _k_vc   = max(1, _ls_vc.selected_orders.get('aic', 1))
            _fit_vc = _VAR_ctrl(_df_vc).fit(_k_vc)
            _roots  = _fit_vc.roots
            _estab  = all(abs(r) < 1 for r in _roots)
            print(f'\n  Mejor VAR (AIC): orden={_k_vc} | '
                  f'estabilidad={"Estable" if _estab else "Inestable"}')
            if not _estab:
                _inest_n = sum(abs(r) >= 1 for r in _roots)
                print(f'    {_inest_n} raices fuera del circulo unitario')
        except Exception as _e_vc:
            print(f'  No se pudo verificar estabilidad: {_e_vc}')

    _n_flag_v = sum(1 for v in _var_flags.values() if v)
    print(f'\n  Total modelos VAR    : {len(df_res_var)}')
    print(f'  Modelos con flags    : {_n_flag_v}')
    print(f'  Modelos sin flags    : {len(df_res_var) - _n_flag_v}')

    df_res_var['ctrl_flags'] = df_res_var['combo_id'].map(
        lambda cid: '|'.join(_var_flags.get(cid, [])) or 'OK'
    )
    df_res_var['ctrl_ok'] = df_res_var['ctrl_flags'] == 'OK'
    df_res_var_ok = df_res_var[df_res_var['ctrl_ok']].copy()
    print(f'\n  df_res_var_ok: {len(df_res_var_ok)} modelos listos para analisis.')
    print('\nControl VAR completado.')


## 9. Modelos VECM

### 9.1 Tests y Transformaciones Previos — Cointegración de Johansen

> El VECM requiere cointegración. El test de Johansen determina el rango global. Si el rango es 0 para una combinación particular, esa combinación se omite del VECM.

In [ ]:
from statsmodels.tsa.vector_ar.vecm import coint_johansen as _joh_s9

print('='*70)
print('9.1  COINTEGRACION DE JOHANSEN (todas las variables disponibles)')
print('='*70)

_N_COINT_GLOBAL = 0
try:
    _data_j9 = df_safe[[Y_SAFE] + X_SAFE_EST].dropna()
    _joh9    = _joh_s9(_data_j9, det_order=0, k_ar_diff=2)
    print(f'  Obs: {len(_data_j9)} | Variables: {len(_data_j9.columns)}')
    print(f'\n  {"r":>4} | {"Trace":>10} {"CV5%":>10} {"Sig":>5} | '
          f'{"MaxEig":>10} {"CV5%":>10} {"Sig":>5}')
    print(f'  {"-"*62}')
    for _ri in range(len(_joh9.lr1)):
        _st = 'OK' if _joh9.lr1[_ri] > _joh9.cvt[_ri, 1] else '  '
        _sm = 'OK' if _joh9.lr2[_ri] > _joh9.cvm[_ri, 1] else '  '
        print(f'  {_ri:>4} | {_joh9.lr1[_ri]:>10.3f} {_joh9.cvt[_ri,1]:>10.3f} {_st:>5} | '
              f'{_joh9.lr2[_ri]:>10.3f} {_joh9.cvm[_ri,1]:>10.3f} {_sm:>5}')
    _N_COINT_GLOBAL = int(sum(_joh9.lr1 > _joh9.cvt[:, 1]))
    print(f'\n  Relaciones de cointegracion globales (Trace, 5%): {_N_COINT_GLOBAL}')
    if _N_COINT_GLOBAL == 0:
        print('  Sin cointegracion global. VECM solo estimable para combos con coint. propia.')
    else:
        print('  VECM aplicable para combinaciones con cointegracion.')
except Exception as _e9:
    _N_COINT_GLOBAL = 0
    print(f'  Error en Johansen global: {_e9}')

print('\nTests de cointegracion completados.')


### 9.2 Estimaciones — VECM

> Estimación iterativa de modelos VECM para las combinaciones de `X_SAFE_EST` que presentan cointegración. Combinaciones sin cointegración son omitidas (no es un error). Los resultados se almacenan en `df_res_vecm`.

In [ ]:
# ── Preparación de combinaciones VECM ────────────────────────────────────────
_n_max_vecm  = len(X_SAFE_EST) if MAX_VARS_POR_MODELO is None else MAX_VARS_POR_MODELO
_combos_vecm = []
for _r in range(1, _n_max_vecm + 1):
    _combos_vecm.extend(list(combinations(X_SAFE_EST, _r)))

if MAX_COMBINACIONES is not None and len(_combos_vecm) > MAX_COMBINACIONES:
    random.seed(SEMILLA)
    _combos_vecm = random.sample(_combos_vecm, MAX_COMBINACIONES)
    print(f'VECM: muestreadas {MAX_COMBINACIONES} de {len(_combos_vecm)} combinaciones')

print(f'Combinaciones VECM : {len(_combos_vecm)}')
print(f'Estimaciones max.  : {len(_combos_vecm)}  (solo con cointegracion)')
print()

_RESULTADOS_VECM = []
_t0_vecm   = datetime.now()
_err_vecm  = 0
_ok_vecm   = 0
_skip_vecm = 0
_PRINT_INT_VECM = max(1, len(_combos_vecm) // 5)

for _ci, _combo in enumerate(_combos_vecm):
    _xv = list(_combo)
    _tv = [Y_SAFE] + _xv
    _df = df_safe[_tv].dropna()
    if len(_df) < 30:
        _skip_vecm += 1
        continue
    try:
        _r = extraer_vecm(_df, Y_SAFE, _xv, _tv, k_ar_diff=2, det_order=0)
        if _r is not None:
            _row = {
                'combo_id'  : f'{_ci}_vecm',
                'criterio'  : 'johansen',
                'modelo'    : 'VECM',
                'n_vars'    : len(_xv),
                'vars_safe' : '|'.join(_xv),
                'vars_label': '|'.join([COL_LABEL[v] for v in _xv]),
                'aic'       : _r['aic'],
                'bic'       : _r['bic'],
                'hq'        : _r['hq'],
                'r2'        : _r['r2'],
                'nobs'      : _r['nobs'],
                'n_coint'   : _r['n_coint'],
            }
            for _x in _xv:
                _row[f'coef_{_x}'] = _r['coefs'][_x]['coef']
                _row[f'pval_{_x}'] = _r['coefs'][_x]['pval']
            _RESULTADOS_VECM.append(_row)
            _ok_vecm += 1
        else:
            _skip_vecm += 1
    except Exception:
        _err_vecm += 1

    if (_ci + 1) % _PRINT_INT_VECM == 0 or _ci == len(_combos_vecm) - 1:
        _el = (datetime.now() - _t0_vecm).seconds
        print(f'  [VECM] combo {_ci+1:>3}/{len(_combos_vecm)} | '
              f'ok={_ok_vecm}  sin_coint={_skip_vecm}  err={_err_vecm}  | {_el}s')

df_res_vecm = pd.DataFrame(_RESULTADOS_VECM)
_tasa_vecm  = f'{_ok_vecm/(_ok_vecm+_err_vecm)*100:.1f}%' if (_ok_vecm+_err_vecm) > 0 else 'n/a'
print(f'\n{"="*60}')
print(f'VECM completado: {_ok_vecm} modelos | {_skip_vecm} sin coint. | {_err_vecm} errores')
print(f'Tasa exito (excl. sin coint.): {_tasa_vecm}')
print(f'{"="*60}')


### 9.3 Control — VECM

> Filtrado de modelos VECM. Se verifica rango de cointegración mínimo y R² mínimo. Los modelos aprobados se almacenan en `df_res_vecm_ok`.

In [ ]:
# ── Control de calidad VECM ──────────────────────────────────────────────────
print('='*70)
print('9.3  CONTROL DE CALIDAD — MODELOS VECM')
print('='*70)

if df_res_vecm.empty:
    print('Sin resultados VECM (ninguna combinacion presento cointegracion).')
    df_res_vecm_ok = pd.DataFrame()
else:
    CTRL_VECM_MIN_R2 = 0.0  # R2 minimo en diferencias (0 = sin filtro)
    CTRL_VECM_MIN_CI = 1    # rango de cointegracion minimo

    _vecm_flags = {row['combo_id']: [] for _, row in df_res_vecm.iterrows()}

    if 'n_coint' in df_res_vecm.columns:
        _low_ci = df_res_vecm[df_res_vecm['n_coint'] < CTRL_VECM_MIN_CI]['combo_id'].tolist()
        for _cid in _low_ci:
            _vecm_flags[_cid].append(f'n_coint_bajo')
        print(f'  Rango coint. < {CTRL_VECM_MIN_CI}: {len(_low_ci)} modelos marcados')

    if CTRL_VECM_MIN_R2 > 0:
        _low_r2v = df_res_vecm[df_res_vecm['r2'] < CTRL_VECM_MIN_R2]['combo_id'].tolist()
        for _cid in _low_r2v:
            _vecm_flags[_cid].append('R2_bajo')
        print(f'  R2 < {CTRL_VECM_MIN_R2}: {len(_low_r2v)} modelos marcados')

    _n_flag_vv = sum(1 for v in _vecm_flags.values() if v)
    print(f'\n  Total modelos VECM   : {len(df_res_vecm)}')
    print(f'  Modelos con flags    : {_n_flag_vv}')
    print(f'  Modelos sin flags    : {len(df_res_vecm) - _n_flag_vv}')

    df_res_vecm['ctrl_flags'] = df_res_vecm['combo_id'].map(
        lambda cid: '|'.join(_vecm_flags.get(cid, [])) or 'OK'
    )
    df_res_vecm['ctrl_ok'] = df_res_vecm['ctrl_flags'] == 'OK'
    df_res_vecm_ok = df_res_vecm[df_res_vecm['ctrl_ok']].copy()
    print(f'\n  df_res_vecm_ok: {len(df_res_vecm_ok)} modelos listos para analisis.')
    print('\nControl VECM completado.')


## Resultados Consolidados

> Une los modelos aprobados por el control de calidad de las secciones 7.3, 8.3 y 9.3 en un único `df_res` y construye el formato largo `df_largo` para los análisis visuales. Ejecutar antes de las secciones 10 y 11.

In [ ]:
# ── Consolidar resultados de los 3 modelos ───────────────────────────────────
_frames_ok = []
for _df_ok, _nombre in [
    (locals().get('df_res_ardl_ok', pd.DataFrame()), 'ARDL'),
    (locals().get('df_res_var_ok',  pd.DataFrame()), 'VAR'),
    (locals().get('df_res_vecm_ok', pd.DataFrame()), 'VECM'),
]:
    if not _df_ok.empty:
        _frames_ok.append(_df_ok)
        print(f'  {_nombre}: {len(_df_ok)} modelos OK')
    else:
        print(f'  {_nombre}: sin modelos (ejecutar seccion correspondiente)')

if not _frames_ok:
    print('\nNo hay modelos consolidados. Ejecute primero las secciones 7, 8 y 9.')
    df_res   = pd.DataFrame()
    df_largo = pd.DataFrame()
else:
    df_res = pd.concat(_frames_ok, ignore_index=True)

    # ── Formato largo ─────────────────────────────────────────────────────────
    _filas = []
    for _, _fila in df_res.iterrows():
        _xv = [v for v in _fila['vars_safe'].split('|') if v]
        for _x in _xv:
            _coef = _fila.get(f'coef_{_x}', float('nan'))
            _pval = _fila.get(f'pval_{_x}', float('nan'))
            import math
            if math.isnan(_coef):
                continue
            _filas.append({
                'combo_id'  : _fila['combo_id'],
                'modelo'    : _fila['modelo'],
                'criterio'  : _fila['criterio'],
                'var_safe'  : _x,
                'variable'  : COL_LABEL.get(_x, _x),
                'coef'      : _coef,
                'pval'      : _pval,
                'sig'       : (_pval < NIVEL_SIG) if not math.isnan(_pval) else False,
                'r2'        : _fila.get('r2', float('nan')),
                'aic'       : _fila.get('aic', float('nan')),
                'bic'       : _fila.get('bic', float('nan')),
                'hq'        : _fila.get('hq',  float('nan')),
                'nobs'      : _fila.get('nobs', float('nan')),
                'n_vars'    : _fila['n_vars'],
                'vars_label': _fila['vars_label'],
            })

    df_largo = pd.DataFrame(_filas)

    print(f'\n{"="*60}')
    print(f'RESULTADOS CONSOLIDADOS')
    print(f'{"="*60}')
    print(f'  df_res   : {len(df_res)} modelos en total')
    for _m in sorted(df_res["modelo"].unique()):
        print(f'    {_m:<8}: {len(df_res[df_res["modelo"]==_m])}')
    print(f'  df_largo : {len(df_largo)} filas x {df_largo.shape[1]} cols')
    print(f'  Variables: {df_largo["variable"].nunique() if not df_largo.empty else 0}')

    _out_path = Path('Variables Regresivas') / 'resultados_iterativos.csv'
    try:
        df_largo.to_csv(_out_path, index=False)
        print(f'\n  Exportado a: {_out_path}')
    except Exception as _e_exp:
        print(f'\n  No se pudo exportar: {_e_exp}')
    print('\nResultados consolidados. Continue con las secciones 10 y 11.')


## 10. Gráficos de Coeficientes, P-valores y Significancia

> Scatter de coeficiente vs p-valor por variable y modelo. Barras de porcentaje de significancia estadística (p < umbral configurado) por variable, por tipo de modelo, y para el total de modelos.

In [ ]:
_COLORS = {'ARDL': '#2196F3', 'VAR': '#FF9800', 'VECM': '#4CAF50'}

if df_largo.empty:
    print('Sin datos para graficar. Ejecute las secciones 7, 8, 9 y Resultados Consolidados.')
else:
    _vars_plot = sorted(df_largo['variable'].unique())

    # ── 10.1 Scatter: Coeficiente vs P-valor por variable ────────────────────
    _nc = min(2, len(_vars_plot))
    _nr = (len(_vars_plot) + _nc - 1) // _nc
    fig, axes = plt.subplots(_nr, _nc, figsize=(8 * _nc, 5 * _nr), squeeze=False)
    for _idx, _var in enumerate(_vars_plot):
        ax  = axes[_idx // _nc][_idx % _nc]
        _dv = df_largo[df_largo['variable'] == _var].dropna(subset=['coef', 'pval'])
        for _mod, _color in _COLORS.items():
            _dm  = _dv[_dv['modelo'] == _mod]
            if _dm.empty:
                continue
            _sig = _dm['sig'].astype(bool)
            ax.scatter(_dm.loc[_sig,  'coef'], _dm.loc[_sig,  'pval'],
                       color=_color, marker='o', label=_mod, alpha=0.7, s=40)
            ax.scatter(_dm.loc[~_sig, 'coef'], _dm.loc[~_sig, 'pval'],
                       color=_color, marker='x', alpha=0.5, s=40)
        ax.axhline(NIVEL_SIG, color='red',  linestyle='--', linewidth=1)
        ax.axvline(0,          color='gray', linestyle='--', linewidth=1)
        ax.set_title(_var[:50], fontsize=9)
        ax.set_xlabel('Coeficiente'); ax.set_ylabel('P-valor')
        ax.legend(fontsize=8)
    for _idx in range(len(_vars_plot), _nr * _nc):
        axes[_idx // _nc][_idx % _nc].set_visible(False)
    fig.suptitle('Coeficiente vs P-valor  (o = sig | x = no sig)', fontsize=12)
    plt.tight_layout()
    plt.show()

    # ── 10.2 % Significancia por variable y modelo ────────────────────────────
    _sig_grp  = (df_largo.groupby(['variable', 'modelo'])['sig']
                 .agg(N='count', N_sig='sum').reset_index())
    _sig_grp['Pct_Sig'] = (_sig_grp['N_sig'] / _sig_grp['N'] * 100).round(1)
    _sig_glob = (df_largo.groupby('variable')['sig']
                 .agg(N='count', N_sig='sum').reset_index())
    _sig_glob['Pct_Sig'] = (_sig_glob['N_sig'] / _sig_glob['N'] * 100).round(1)
    _sig_glob['modelo']  = 'TOTAL'
    _sig_all  = pd.concat([_sig_grp, _sig_glob], ignore_index=True)
    _order    = _sig_glob.sort_values('Pct_Sig', ascending=False)['variable'].tolist()
    _modelos  = [m for m in ['ARDL', 'VAR', 'VECM']
                 if m in _sig_grp['modelo'].unique()] + ['TOTAL']
    _colors_e = {**_COLORS, 'TOTAL': '#9C27B0'}
    _x_pos    = np.arange(len(_order))
    _bw       = 0.8 / len(_modelos)

    fig, ax = plt.subplots(figsize=(max(10, len(_order) * 1.2), 6))
    for _i, _mod in enumerate(_modelos):
        _dm   = _sig_all[_sig_all['modelo'] == _mod].set_index('variable')
        _vals = [_dm.loc[v, 'Pct_Sig'] if v in _dm.index else 0 for v in _order]
        ax.bar(_x_pos + _i * _bw, _vals, width=_bw,
               color=_colors_e[_mod], label=_mod, alpha=0.85)
    ax.set_xticks(_x_pos + _bw * (len(_modelos) - 1) / 2)
    ax.set_xticklabels([v[:30] for v in _order], rotation=45, ha='right', fontsize=8)
    ax.axhline(50, color='red', linestyle='--', linewidth=0.8, label='50%')
    ax.set_ylabel(f'% veces significativa (p < {NIVEL_SIG})')
    ax.set_title('Porcentaje de significancia por variable y modelo')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    # ── Tabla resumen: % Sig y Coeficiente Promedio ───────────────────────────
    _pct_piv  = (_sig_grp.pivot(index='variable', columns='modelo', values='Pct_Sig')
                 .rename_axis(None, axis=1))
    _pct_piv['TOTAL'] = _sig_glob.set_index('variable')['Pct_Sig']
    _pct_piv  = _pct_piv.loc[_order]
    _coef_piv = (df_largo.groupby(['variable', 'modelo'])['coef']
                 .mean().unstack().round(4).rename_axis(None, axis=1))
    display(pd.concat({'% Sig': _pct_piv, 'Coef. Prom.': _coef_piv}, axis=1))


## 11. Ranking y Comparación de Modelos

> Top 10 modelos por cada criterio de información (R², AIC, BIC, HQ). Resultados del mejor modelo global y gráficos de comparación entre modelos. Requiere que `df_res` y `df_largo` estén definidos (ejecutar "Resultados Consolidados" primero).

In [ ]:
_COLORS = {'ARDL': '#2196F3', 'VAR': '#FF9800', 'VECM': '#4CAF50'}

if df_res.empty:
    print('Sin resultados para mostrar. Ejecute las secciones 7, 8 y 9 primero.')
else:
    # ── 11.1 Top 10 por cada criterio ────────────────────────────────────────
    _criterios_rank = {
        'R2'  : ('r2',  False),
        'AIC' : ('aic', True),
        'BIC' : ('bic', True),
        'HQ'  : ('hq',  True),
    }
    for _crit_name, (_col, _asc) in _criterios_rank.items():
        if _col not in df_res.columns or df_res[_col].isna().all():
            continue
        print(f'\n{"="*70}')
        print(f'TOP 10 por {_crit_name}  ({"menor" if _asc else "mayor"} = mejor)')
        print('='*70)
        _top = (
            df_res.dropna(subset=[_col])
            .sort_values(_col, ascending=_asc)
            .drop_duplicates(subset=['vars_safe', 'modelo'])
            .head(10)
            [['modelo', 'vars_label', _col, 'aic', 'bic', 'hq', 'r2', 'nobs']]
            .reset_index(drop=True)
        )
        _top.index = range(1, len(_top) + 1)
        display(_top.rename(columns={'vars_label': 'Variables', 'modelo': 'Modelo'}))

    # ── 11.2 Mejor modelo global (menor AIC) ─────────────────────────────────
    print(f'\n{"="*70}')
    print('MEJOR MODELO GLOBAL (menor AIC)')
    print('='*70)
    _best_g  = df_res.loc[df_res['aic'].idxmin()]
    _xv_g    = [v for v in _best_g['vars_safe'].split('|') if v]
    _xl_g    = [COL_LABEL.get(v, v) for v in _xv_g]
    print(f'\n  Modelo : {_best_g["modelo"]}')
    print(f'  AIC={_best_g["aic"]:.4f}  BIC={_best_g["bic"]:.4f}  '
          f'HQ={_best_g["hq"]:.4f}  R2={_best_g.get("r2", float("nan")):.4f}')
    print(f'  Obs={int(_best_g["nobs"])}  N_vars={int(_best_g["n_vars"])}')
    print(f'  Variables: {", ".join(_xl_g)}')
    print('\n  Coeficientes:')
    for _x, _xl in zip(_xv_g, _xl_g):
        _c = _best_g.get(f'coef_{_x}', float('nan'))
        _p = _best_g.get(f'pval_{_x}', float('nan'))
        import math
        if math.isnan(_c):
            continue
        _s = '***' if _p < 0.01 else '**' if _p < 0.05 else '*' if _p < 0.10 else ''
        print(f'    {_xl[:50]:<50}  coef={_c:+.4f}  p={_p:.4f} {_s}')
    print('\n  * p<10%  ** p<5%  *** p<1%')

    # ── 11.3 Box plots AIC / BIC / HQ ────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, crit in zip(axes, ['aic', 'bic', 'hq']):
        _data_bp = [df_res[df_res['modelo'] == m][crit].dropna().values
                    for m in _COLORS if m in df_res['modelo'].unique()]
        _labels  = [m for m in _COLORS if m in df_res['modelo'].unique()]
        if not _data_bp:
            continue
        bp = ax.boxplot(_data_bp, labels=_labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], [_COLORS[m] for m in _labels]):
            patch.set_facecolor(color); patch.set_alpha(0.7)
        ax.set_title(crit.upper())
    fig.suptitle('Distribucion de criterios de informacion por modelo')
    plt.tight_layout()
    plt.show()

    # ── 11.4 AIC por numero de variables ──────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4))
    for _mod, _color in _COLORS.items():
        if _mod not in df_res['modelo'].unique():
            continue
        _dm = df_res[df_res['modelo'] == _mod].groupby('n_vars')['aic'].agg(['mean', 'min'])
        ax.plot(_dm.index, _dm['mean'], color=_color, linewidth=2, marker='o',
                label=f'{_mod} media')
        ax.plot(_dm.index, _dm['min'],  color=_color, linewidth=1, linestyle=':',
                label=f'{_mod} min')
    ax.set_title('AIC por cantidad de variables explicativas')
    ax.set_xlabel('N vars'); ax.set_ylabel('AIC'); ax.legend()
    plt.tight_layout()
    plt.show()

    # ── 11.5 Real vs Ajustado — mejor modelo global ───────────────────────────
    _best_mod = _best_g['modelo']
    _tv_best  = [Y_SAFE] + _xv_g
    _df_best  = df_safe[_tv_best].dropna()
    try:
        if _best_mod == 'ARDL':
            from statsmodels.tsa.ardl import ARDL as _ARDL_plt
            _bj_lp    = globals().get('_BJ_PROPOSED_LAGS', {})
            _bj_ary   = int(globals().get('_BJ_AR_Y', 1))
            _ord_d    = {x: max(_bj_lp.get(x, [0, 1])) for x in _xv_g}
            _fit_best = _ARDL_plt(_df_best[Y_SAFE], lags=_bj_ary,
                                   exog=_df_best[_xv_g], order=_ord_d, trend='c').fit()
            _fitted_b = _fit_best.fittedvalues
            _actual_b = _df_best[Y_SAFE]
            _label_b  = 'ARDL ajustado'
        elif _best_mod == 'VAR':
            from statsmodels.tsa.api import VAR as _VAR_plt2
            _ls_b  = _VAR_plt2(_df_best).select_order(maxlags=MAX_LAGS_VAR)
            _kb    = max(1, _ls_b.selected_orders.get('aic', 1))
            _fit_b = _VAR_plt2(_df_best).fit(_kb)
            _yidx  = list(_tv_best).index(Y_SAFE)
            _fitted_b = _fit_b.fittedvalues.iloc[:, _yidx]
            _actual_b = _df_best[Y_SAFE].iloc[_kb:]
            _label_b  = f'VAR({_kb}) ajustado'
        elif _best_mod == 'VECM':
            from statsmodels.tsa.vector_ar.vecm import (VECM as _VECM_plt2,
                                                         coint_johansen as _joh_plt2)
            _joh_b = _joh_plt2(_df_best, det_order=0, k_ar_diff=2)
            _nci_b = int(sum(_joh_b.lr1 > _joh_b.cvt[:, 1]))
            if _nci_b == 0:
                raise ValueError('Sin cointegracion en el mejor modelo')
            _fit_b    = _VECM_plt2(_df_best, k_ar_diff=2, coint_rank=_nci_b,
                                    deterministic='ci').fit()
            _yidx     = list(_tv_best).index(Y_SAFE)
            _actual_b = _df_best[Y_SAFE].diff().dropna()
            _fitted_b = pd.Series(
                _fit_b.fittedvalues[:, _yidx],
                index=_actual_b.index[-len(_fit_b.fittedvalues):]
            )
            _label_b = f'VECM(r={_nci_b}) ajustado'

        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(_actual_b.index, _actual_b.values,
                color='#1f77b4', linewidth=1.5, label='Real')
        ax.plot(_fitted_b.index, _fitted_b.values,
                color='#FF9800', linewidth=1.5, linestyle='--', label=_label_b)
        ax.set_title(f'Mejor modelo global ({_best_mod}) — Real vs Ajustado | Y = {ETI_DEP[:50]}')
        ax.legend(); ax.set_ylabel(ETI_DEP[:40])
        plt.tight_layout()
        plt.show()
    except Exception as _e_plt:
        print(f'No se pudo graficar el ajuste: {_e_plt}')


## 12.5  Resultados del Mejor VAR y VECM
> Ajustado vs real y **Funciones de Respuesta al Impulso (FRI)** para la especificacion con menor AIC de cada modelo.

In [ ]:
# ===========================================================================
# 12.5  Visualizacion del mejor VAR y VECM
# ===========================================================================
_IRF_PERIODS = 12   # horizonte para funciones de impulso-respuesta

def _get_best_spec(modelo):
    _dm = df_res[df_res['modelo'] == modelo]
    if _dm.empty:
        return None
    return _dm.loc[_dm['aic'].idxmin()]

# ===========================================================================
# VAR -- ajustado vs real + FRI
# ===========================================================================
_best_var = _get_best_spec('VAR')
if _best_var is None:
    print('Sin resultados VAR para graficar.')
else:
    from statsmodels.tsa.api import VAR as _VAR_plot
    _xv_var     = [v for v in _best_var['vars_safe'].split('|') if v]
    _tv_var     = [Y_SAFE] + _xv_var
    _df_var     = df_safe[_tv_var].dropna()
    _ls_var     = _VAR_plot(_df_var).select_order(maxlags=MAX_LAGS_VAR)
    _k_var      = max(1, _ls_var.selected_orders.get('aic', 1))
    _fit_var    = _VAR_plot(_df_var).fit(_k_var)
    _y_idx_var  = list(_tv_var).index(Y_SAFE)
    _xv_lab_var = [COL_LABEL.get(v, v) for v in _xv_var]
    print(f'Mejor VAR: orden={_k_var}  |  vars: {", ".join(_xv_lab_var)}')
    print(f'  AIC={_best_var["aic"]:.4f}  BIC={_best_var["bic"]:.4f}')

    # Ajustado vs Real
    _fitted_var = _fit_var.fittedvalues.iloc[:, _y_idx_var]
    _actual_var = _df_var[Y_SAFE].iloc[_k_var:]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(_actual_var.index, _actual_var.values,
            color='#1f77b4', linewidth=1.5, label='Real')
    ax.plot(_fitted_var.index, _fitted_var.values,
            color='#FF9800', linewidth=1.5, linestyle='--',
            label=f'Ajustado VAR({_k_var})')
    ax.set_title(f'VAR({_k_var}) — Real vs Ajustado  |  Y = {ETI_DEP[:60]}')
    ax.legend(); ax.set_ylabel(ETI_DEP[:40])
    plt.tight_layout()
    plt.show()

    # Funciones de Impulso-Respuesta
    try:
        _irf_var = _fit_var.irf(_IRF_PERIODS)
        _irf_var.plot(impulse=Y_SAFE, response=Y_SAFE,
                      title=f'FRI VAR({_k_var}) — impulso: {ETI_DEP[:40]}')
        plt.tight_layout()
        plt.show()
    except Exception as _e_irf:
        print(f'FRI VAR no disponible: {_e_irf}')

# ===========================================================================
# VECM -- ajustado vs real + vector de cointegracion + FRI
# ===========================================================================
_best_vecm = _get_best_spec('VECM')
if _best_vecm is None:
    print('\nSin resultados VECM (ninguna combinacion presento cointegracion).')
else:
    from statsmodels.tsa.vector_ar.vecm import (VECM as _VECM_plot,
                                                 coint_johansen as _joh_v2)
    _xv_vecm     = [v for v in _best_vecm['vars_safe'].split('|') if v]
    _tv_vecm     = [Y_SAFE] + _xv_vecm
    _df_vecm     = df_safe[_tv_vecm].dropna()
    _joh_v       = _joh_v2(_df_vecm, det_order=0, k_ar_diff=2)
    _nci         = int(sum(_joh_v.lr1 > _joh_v.cvt[:, 1]))
    _xv_lab_vecm = [COL_LABEL.get(v, v) for v in _xv_vecm]
    print(f'\nMejor VECM: rango={_nci}  |  vars: {", ".join(_xv_lab_vecm)}')
    print(f'  AIC={_best_vecm["aic"]:.4f}  BIC={_best_vecm["bic"]:.4f}')

    if _nci == 0:
        print('Sin cointegracion en la mejor especificacion VECM.')
    else:
        _fit_vecm = _VECM_plot(_df_vecm, k_ar_diff=2, coint_rank=_nci,
                               deterministic='ci').fit()
        _y_idx_v = list(_tv_vecm).index(Y_SAFE)

        # Ajustado vs Real (en diferencias)
        _actual_dv = _df_vecm[Y_SAFE].diff().dropna()
        _fitted_dv = pd.Series(
            _fit_vecm.fittedvalues[:, _y_idx_v],
            index=_actual_dv.index[-len(_fit_vecm.fittedvalues):]
        )
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(_actual_dv.index, _actual_dv.values,
                color='#1f77b4', linewidth=1.5, label='ΔReal')
        ax.plot(_fitted_dv.index, _fitted_dv.values,
                color='#4CAF50', linewidth=1.5, linestyle='--',
                label='ΔVECM ajustado')
        ax.set_title(
            f'VECM(rango={_nci}) — Real vs Ajustado (diferencias)  |  Y = {ETI_DEP[:50]}'
        )
        ax.legend(); ax.set_ylabel(f'Δ{ETI_DEP[:35]}')
        plt.tight_layout()
        plt.show()

        # Vector de cointegracion
        print(f'\n  Vector de cointegracion (rango={_nci}):')
        _beta_df = pd.DataFrame(
            _fit_vecm.beta,
            index=[COL_LABEL.get(v, v) for v in _tv_vecm],
            columns=[f'CE{i+1}' for i in range(_nci)]
        )
        display(_beta_df.round(4))

        # Velocidad de ajuste (alpha) para la ecuacion Y
        print(f'\n  Velocidad de ajuste α para "{ETI_DEP[:50]}":')
        _alpha_row = _fit_vecm.alpha[_y_idx_v]
        for _ci_i, _a in enumerate(_alpha_row):
            if _a < 0:
                _vida = abs(np.log(2) / _a)
                print(f'    CE{_ci_i+1}: α = {_a:.4f}  (½-vida ≈ {_vida:.1f} periodos)')
            else:
                print(f'    CE{_ci_i+1}: α = {_a:.4f}  (positivo — sin convergencia al equilibrio)')

        # FRI VECM
        try:
            _irf_vecm = _fit_vecm.irf(_IRF_PERIODS)
            _irf_vecm.plot(impulse=Y_SAFE, response=Y_SAFE,
                           title=f'FRI VECM(rango={_nci}) — impulso: {ETI_DEP[:40]}')
            plt.tight_layout()
            plt.show()
        except Exception as _e_irf_v:
            print(f'FRI VECM no disponible: {_e_irf_v}')


## Apéndice A — Explorador Interactivo de Resultados

> Tabla interactiva para explorar `df_largo`. Requiere la librería `itables`.

In [ ]:
try:
    from itables import show as _it_show, init_notebook_mode as _it_init
except ImportError:
    import subprocess as _sp_it, sys as _sys_it
    _sp_it.check_call([_sys_it.executable, '-m', 'pip', 'install', 'itables', '-q'])
    from itables import show as _it_show, init_notebook_mode as _it_init

_it_init(all_tables=False)

# ── Preparar tabla de visualización ──────────────────────────────────────────
_df_view = df_largo[[
    'modelo', 'criterio', 'variable', 'coef', 'pval', 'sig',
    'r2', 'aic', 'bic', 'hq', 'nobs', 'n_vars', 'vars_label'
]].copy()

_df_view['coef'] = _df_view['coef'].round(4)
_df_view['pval'] = _df_view['pval'].round(4)
_df_view['r2']   = _df_view['r2'].round(4)
_df_view['aic']  = _df_view['aic'].round(2)
_df_view['bic']  = _df_view['bic'].round(2)
_df_view['hq']   = _df_view['hq'].round(2)
_df_view['sig']  = _df_view['sig'].map({True: '✅ sí', False: '❌ no'})
_df_view = _df_view.sort_values(['modelo', 'variable', 'pval']).reset_index(drop=True)

print(f'Total de filas: {len(_df_view)}  — podés filtrar por cualquier columna.')
_it_show(
    _df_view,
    caption  = f'Resultados completos — Y = {ETI_DEP}',
    maxBytes = 0,           # sin límite de tamaño
    lengthMenu = [25, 50, 100, 200, 500],
    style    = 'width:100%; font-size:12px',
    classes  = 'display compact hover',
)


## Apéndice B — Diagnóstico de Residuos

> Test Ljung-Box de ruido blanco sobre los residuos de los mejores modelos.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox as _lb_test

_LB_LAGS = 12   # lags para Ljung-Box

def _resid_para_modelo(row):
    modelo = row['modelo']
    xv     = [v for v in str(row['vars_safe']).split('|') if v]
    tv     = [Y_SAFE] + xv
    df_m   = df_safe[tv].dropna()
    if len(df_m) < 30:
        return None
    try:
        if modelo == 'ARDL':
            from statsmodels.tsa.ardl import ARDL as _ARDL_r
            _bj_ok = '_BJ_PROPOSED_LAGS' in globals()
            if _bj_ok:
                try:
                    _bj_lp  = globals()['_BJ_PROPOSED_LAGS']
                    _bj_ary = int(globals().get('_BJ_AR_Y', 1))
                    _od     = {x: max(_bj_lp.get(x, [0, 1])) for x in xv}
                    _fit    = _ARDL_r(df_m[Y_SAFE], lags=_bj_ary, exog=df_m[xv],
                                      order=_od, trend='c').fit()
                except Exception:
                    _bj_ok = False
            if not _bj_ok:
                from statsmodels.tsa.ardl import ardl_select_order as _aso
                _sel = _aso(df_m[Y_SAFE], MAX_LAGS_DEP, df_m[xv],
                            MAX_LAGS_IND, ic='aic', trend='c')
                _fit = _sel.model.fit()
            return _fit.resid.dropna().values

        elif modelo == 'VAR':
            from statsmodels.tsa.api import VAR as _VAR_r
            _ls  = _VAR_r(df_m).select_order(maxlags=MAX_LAGS_VAR)
            _kar = max(1, _ls.selected_orders.get('aic', 1))
            _fit = _VAR_r(df_m).fit(_kar)
            return _fit.resid[:, list(tv).index(Y_SAFE)]

        elif modelo == 'VECM':
            from statsmodels.tsa.vector_ar.vecm import VECM as _VECM_r, coint_johansen as _joh_r
            _joh = _joh_r(df_m, det_order=0, k_ar_diff=2)
            _nci = int(sum(_joh.lr1 > _joh.cvt[:, 1]))
            if _nci == 0:
                return None
            _fit = _VECM_r(df_m, k_ar_diff=2, coint_rank=_nci,
                           deterministic='ci').fit()
            return _fit.resid[:, list(tv).index(Y_SAFE)]
    except Exception:
        return None

# De-duplicar: una estimación por (vars_safe, modelo) — se usa la de mejor AIC
_df_uniq_rb = (
    df_res.sort_values('aic')
    .groupby(['vars_safe', 'modelo'], sort=False)
    .first()
    .reset_index()
)

print(f'Testeando {len(_df_uniq_rb)} especificaciones únicas '
      f'(Ljung-Box, {_LB_LAGS} lags, α={NIVEL_SIG})...\n')

_wb_rows = []
for _i_rb, _row_rb in _df_uniq_rb.iterrows():
    _resid = _resid_para_modelo(_row_rb)
    if _resid is None or len(_resid) < 15:
        continue
    try:
        _lb    = _lb_test(_resid, lags=[_LB_LAGS], return_df=True)
        _lbst  = float(_lb['lb_stat'].iloc[0])
        _lbpv  = float(_lb['lb_pvalue'].iloc[0])
        _es_rb = _lbpv >= NIVEL_SIG
        _xv_rb = [v for v in str(_row_rb['vars_safe']).split('|') if v]
        _wb_rows.append({
            'Modelo'       : _row_rb['modelo'],
            'N_vars'       : int(_row_rb['n_vars']),
            'Variables'    : ' | '.join([COL_LABEL.get(v, v)[:22] for v in _xv_rb]),
            'LB_stat'      : round(_lbst, 3),
            'LB_pval'      : round(_lbpv, 4),
            'Ruido_Blanco' : '✅ Sí' if _es_rb else '❌ No',
            '_pass'        : _es_rb,
        })
    except Exception:
        pass

    if (_i_rb + 1) % 30 == 0 or _i_rb == len(_df_uniq_rb) - 1:
        print(f'  {_i_rb+1:>3}/{len(_df_uniq_rb)} completados — '
              f'fallas hasta ahora: {sum(not r["_pass"] for r in _wb_rows)}')

df_rb = pd.DataFrame(_wb_rows)
print(f'\n✅ {len(df_rb)} tests completados\n')

# ── Resumen por tipo de modelo ────────────────────────────────────────────────
print('═'*70)
print('RESUMEN POR TIPO DE MODELO')
print('═'*70)
_rb_res = (
    df_rb.groupby('Modelo')
    .agg(Total=('_pass', 'count'), RB_ok=('_pass', 'sum'))
    .assign(
        No_RB    = lambda d: d['Total'] - d['RB_ok'],
        Pct_OK   = lambda d: (d['RB_ok'] / d['Total'] * 100).round(1),
        Pct_NoRB = lambda d: (d['No_RB'] / d['Total'] * 100).round(1),
    )
    .rename(columns={
        'RB_ok' : '✅ Ruido Blanco',
        'No_RB' : '❌ No Ruido Blanco',
        'Pct_OK': '% OK',
        'Pct_NoRB': '% No OK',
    })
)
display(_rb_res)

# ── Tabla completa (fallas primero) ──────────────────────────────────────────
print('\n' + '═'*70)
print(f'TABLA COMPLETA  (ordenada: fallas primero | Ljung-Box {_LB_LAGS} lags)')
print('═'*70)
_df_rb_show = (
    df_rb.drop(columns='_pass')
    .sort_values(['Ruido_Blanco', 'Modelo', 'LB_pval'])
    .reset_index(drop=True)
)
display(_df_rb_show)

# ── Solo los que NO pasan ─────────────────────────────────────────────────────
_no_rb = df_rb[~df_rb['_pass']].drop(columns='_pass').reset_index(drop=True)
print(f'\n  Modelos cuyos errores NO son ruido blanco: {len(_no_rb)} / {len(df_rb)}')
if not _no_rb.empty:
    print('  (autocorrelación residual detectada — revisar especificación o lags)')
    display(_no_rb)
else:
    print('  Todos los modelos estimados pasan el test de Ljung-Box. ✅')